# Qwen3-ASR A2S Fast Fine-tune

唯一执行路径：公开 gold transcript、单 BF16 LoRA adapter、先 smoke 再完整训练。

### 运行前提

- 使用支持 BF16 的 Colab GPU，正式训练建议 A100 运行时。
- Google Drive 用于保存候选清单、manifest、checkpoint 和评测结果；音频放在 `/content` 本地盘以提高训练吞吐。
- 首次执行必须从环境初始化开始，并严格按 1 到 6 的顺序运行。第 1、2 阶段未通过时，不要启动完整数据 staging。
- 推理、staging 和训练都支持恢复。运行时中断后重新执行对应单元即可，脚本会跳过已持久化的有效结果。

### 结果目录

所有持久化产物写入 Drive 的 `qwen3-asr-public-a2s/`：`candidates/` 保存候选行，`manifests/` 保存固定数据集，`base/` 保存基线，`runs/` 保存训练状态，`results/` 保存最终评测。

In [ ]:
# Drive 保存可恢复产物；仓库本身使用当前远端 main。
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
PROJECT = Path('/content/mega-asr')
if not (PROJECT / '.git').is_dir():
    # 第一次运行只克隆独立 Qwen3-ASR 项目。
    subprocess.run(['git', 'clone', 'https://github.com/cluster1900/lora-asr.git', str(PROJECT)], check=True)
else:
    # 已存在仓库只允许 fast-forward，避免混入 Colab 本地改动。
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(PROJECT)
print('repo_commit=', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# 安装固定依赖；保留 Colab 自带的 CUDA PyTorch。
%pip install -q -r requirements-colab.txt
# FlashAttention 单独安装，并复用当前运行时已有的 CUDA 工具链。
%pip install -q flash-attn==2.8.3.post1 --no-build-isolation

In [ ]:
# Drive 目录在运行时重启后仍存在；LOCAL_DATA 位于临时本地 SSD。
DRIVE_ROOT = Path('/content/drive/MyDrive/qwen3-asr-public-a2s')
LOCAL_DATA = Path('/content/qwen3-asr-runtime/data')
SMOKE_CANDIDATES = DRIVE_ROOT / 'candidates/smoke'
FULL_CANDIDATES = DRIVE_ROOT / 'candidates/full'
MANIFESTS = DRIVE_ROOT / 'manifests'
BASE = DRIVE_ROOT / 'base'
RESULTS = DRIVE_ROOT / 'results'
SMOKE_RUN = DRIVE_ROOT / 'runs/smoke-v1'
for path in (LOCAL_DATA, SMOKE_CANDIDATES, FULL_CANDIDATES, MANIFESTS, BASE, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

DATA_CONFIG = PROJECT / 'configs/data/public_robust_200k.yaml'
TRAIN_CONFIG = PROJECT / 'configs/train/qwen3_asr_public_200k_a2s.yaml'
DATA_TOOL = PROJECT / 'scripts/prepare_public_robust_manifests.py'
INFER = PROJECT / 'inference/qwen3_asr_infer.py'
EVAL = PROJECT / 'evaluation/eval_wer.py'
TRAIN = PROJECT / 'train/train_qwen3_asr_a2s.py'

def run(*parts, check=True):
    """显示并执行仓库 CLI；check=False 仅用于预期可能配额不足的探测。"""
    command = [str(part) for part in parts]
    print('$', ' '.join(command))
    return subprocess.run(command, cwd=PROJECT, check=check, text=True)

## 1. Metadata 与 128-row smoke

**目的**：用最小真实音频闭环验证远端 schema、固定 revision、音频解码、manifest 配额、基础模型推理和双语评测。

**产物**：`probe.json`、smoke candidates、`public_robust_smoke_128.jsonl`、4-row Bench smoke、base predictions 和 metrics。

**通过标准**：命令均以 0 退出；128 条训练行可解码且无硬泄漏；基础模型至少成功处理一条 clean 和一条 degraded 音频。失败时停在本阶段，检查 validation/rejects 报告。

In [ ]:
# probe 只读取元数据，不下载音频；用于尽早发现数据集 revision/schema 漂移。
run('python', DATA_TOOL, 'probe', '--config', DATA_CONFIG, '--output', DRIVE_ROOT / 'probe.json')
# smoke staging 按语言和场景取最小配额，并逐条保存音频与候选 JSONL。
run('python', DATA_TOOL, 'stage', '--config', DATA_CONFIG, '--mode', 'smoke',
    '--candidate-dir', SMOKE_CANDIDATES, '--data-root', LOCAL_DATA)
# 从本地候选构造固定 smoke manifest；decode 门禁会实际读取每个音频文件。
run('python', DATA_TOOL, 'smoke', '--config', DATA_CONFIG,
    '--robust-candidates', SMOKE_CANDIDATES / 'robust.jsonl',
    '--english-clean-candidates', SMOKE_CANDIDATES / 'english_clean.jsonl',
    '--chinese-clean-candidates', SMOKE_CANDIDATES / 'chinese_clean.jsonl',
    '--bench-candidates', SMOKE_CANDIDATES / 'bench.jsonl',
    '--output-dir', MANIFESTS, '--data-root', LOCAL_DATA, '--audio-mode', 'decode', '--force')

In [ ]:
SMOKE_MANIFEST = MANIFESTS / 'public_robust_smoke_128.jsonl'
SMOKE_PREDICTIONS = BASE / 'smoke_128.predictions.jsonl'
# --resume 按 sample_id 跳过已落盘预测，断线后可以安全重跑本单元。
run('python', INFER, '--manifest', SMOKE_MANIFEST, '--output-jsonl', SMOKE_PREDICTIONS,
    '--audio-root', LOCAL_DATA, '--resume')
run('python', EVAL, '--predictions-jsonl', SMOKE_PREDICTIONS, '--output-dir', BASE / 'smoke_128')

import json
# 基线必须同时产生 clean 与 degraded 有效输出，否则不得进入训练 smoke。
rows = [json.loads(line) for line in SMOKE_PREDICTIONS.read_text(encoding='utf-8').splitlines() if line]
successful = [row for row in rows if not row.get('error')]
assert any(row.get('condition_group') == 'clean' for row in successful)
assert any(row.get('condition_group') in {'atomic', 'compound'} for row in successful)

## 2. 10+2 step checkpoint/resume 门禁

**目的**：先静态确认 343-target 合同，再用 10 step 保存 checkpoint，并由第二个训练进程恢复到 12 step。这里验证的是训练链可靠性，不判断模型效果。

**产物**：`runs/smoke-v1/` 下的 checkpoint、final adapter、Trainer state、resolved contract 和 smoke result。

**通过标准**：target 分组为 96/48/3/112/84；loss、gradient、learning rate 有限；第二次运行从 step 10 连续到 step 12。OOM 时只调整 micro batch 与 gradient accumulation，effective batch 保持 128。

In [ ]:
# validate-only 不加载 GPU 模型，用固定合同先拦截配置错误。
run('python', TRAIN, '--config', TRAIN_CONFIG, '--validate-only', '--print-plan')
# 第一个进程训练 10 step 并保存 checkpoint。
run('python', TRAIN, '--config', TRAIN_CONFIG, '--output-dir', SMOKE_RUN, '--smoke-steps', '10')
# 第二个进程自动找到最新 checkpoint，并把总 global step 继续到 12。
run('python', TRAIN, '--config', TRAIN_CONFIG, '--output-dir', SMOKE_RUN,
    '--smoke-steps', '12', '--resume', 'auto')

## 3. 按配额 staging 完整数据

**目的**：仅在前两关通过后，物化正式训练、验证和 Bench 所需公开音频，并生成固定 manifest。

**产物**：200k train、10k validation、512 canary、5k Bench、来源快照、统计、rejects 和 validation report。

**恢复说明**：staging 每 100 行持久化；重跑会校验已有文件 hash，只补缺失或损坏行。Colab 运行时重启会清空 `/content` 音频，因此需要重新执行本阶段恢复音频。

In [ ]:
# full staging 按固定配额停止，不遍历或下载数据集的无关尾部。
run('python', DATA_TOOL, 'stage', '--config', DATA_CONFIG, '--mode', 'full',
    '--candidate-dir', FULL_CANDIDATES, '--data-root', LOCAL_DATA)
# build 只消费已物化候选，并执行计数、路径、解码和跨切分泄漏硬门禁。
run('python', DATA_TOOL, 'build', '--config', DATA_CONFIG,
    '--robust-candidates', FULL_CANDIDATES / 'robust.jsonl',
    '--english-clean-candidates', FULL_CANDIDATES / 'english_clean.jsonl',
    '--chinese-clean-candidates', FULL_CANDIDATES / 'chinese_clean.jsonl',
    '--bench-candidates', FULL_CANDIDATES / 'bench.jsonl',
    '--output-dir', MANIFESTS, '--data-root', LOCAL_DATA, '--audio-mode', 'decode', '--force')

## 4. 最少量 BF16 base 评分并生成 30k curriculum

**目的**：用同一固定 BF16 base 对 gold transcript 评分，选择 `base_error_rate < 0.70` 的 30k 训练行，并生成由易到难的累计视图。base prediction 只用于排序，不作为训练标签。

**节省时间策略**：先评 60k；可用行不足时才依次扩到 100k、160k、200k。预测文件持续恢复，不会重复推理。

**产物**：`public_robust_30k_curriculum.jsonl`、阈值视图、评分 JSONL 和 curriculum report。

In [ ]:
TRAIN_MANIFEST = MANIFESTS / 'public_robust_200k_train.jsonl'
BASE_TRAIN_PREDICTIONS = BASE / 'train_curriculum.predictions.jsonl'
CURRICULUM = MANIFESTS / 'public_robust_30k_curriculum.jsonl'
# 每轮扩大上限时 --resume 只推理新增区间，避免重复消耗 GPU。
for limit in (60000, 100000, 160000, 200000):
    run('python', INFER, '--manifest', TRAIN_MANIFEST, '--output-jsonl', BASE_TRAIN_PREDICTIONS,
        '--audio-root', LOCAL_DATA, '--limit', limit, '--resume')
    run('python', EVAL, '--predictions-jsonl', BASE_TRAIN_PREDICTIONS,
        '--output-dir', BASE / 'train_curriculum')
    # 配额不足是本循环的预期信号，因此暂不让 subprocess 立即抛错。
    built = run('python', DATA_TOOL, 'curriculum', '--config', DATA_CONFIG,
        '--train', TRAIN_MANIFEST, '--scored', BASE / 'train_curriculum/scored.jsonl',
        '--output', CURRICULUM, '--report', MANIFESTS / 'public_robust_30k_curriculum.report.json',
        '--force', check=False)
    if built.returncode == 0:
        print('curriculum_ready_at_limit=', limit)
        break
else:
    raise RuntimeError('200k base predictions still cannot produce the fixed 30k curriculum')

## 5. 固定 base canary、validation 与 Bench baseline

**目的**：在训练前冻结三份 base 指标。512 canary 用于每阶段安全门禁；10k validation 与 5k Bench 用于最终同口径对比。

**产物**：每个集合的原始 predictions、scored JSONL、metrics JSON 与按语言/场景/cell CSV。中英文分别读取 WER/CER，不使用混合 error rate。

In [ ]:
# 三份基线都使用完全相同的模型 revision、BF16 dtype 和解码上限。
BASE_JOBS = {
    'validation_canary_512': MANIFESTS / 'public_robust_512_canary.jsonl',
    'validation_10k': MANIFESTS / 'public_robust_10k_val.jsonl',
    'bench_5k': MANIFESTS / 'vitw_bench_5k_test.jsonl',
}
for name, manifest in BASE_JOBS.items():
    # 每个集合独立落盘，任一长任务中断后可从已有 sample_id 继续。
    predictions = BASE / f'{name}.predictions.jsonl'
    run('python', INFER, '--manifest', manifest, '--output-jsonl', predictions,
        '--audio-root', LOCAL_DATA, '--resume')
    run('python', EVAL, '--predictions-jsonl', predictions, '--output-dir', BASE / name)

## 6. 单 adapter 三阶段训练与 release 评测

**目的**：按 Phase I（上层 audio/projection）→ Phase II（decoder）→ Phase III（全部 target）训练同一个 adapter。每阶段结束自动运行 512 canary；门禁不通过会立即停止。

**产物**：阶段 checkpoint/final adapter、canary predictions/metrics/gate、release adapter/processor/manifest，以及最终 validation 和 Bench 结果。

**最终验收**：release 可由新进程加载；相对 base 报告中英文 WER/CER、clean regression、robust macro、空输出、重复、过长、幻觉式输出和推理失败率。

In [ ]:
# --resume auto 只恢复未完成阶段；已通过 canary 的阶段由 pipeline_state 跳过。
run('python', TRAIN, '--config', TRAIN_CONFIG, '--resume', 'auto')
ADAPTER = DRIVE_ROOT / 'runs/main/release/adapter'
assert ADAPTER.is_dir(), ADAPTER
# 最终 adapter 只评一次固定 validation 与 Bench，并与上一步 base 结果同口径比较。
for name in ('validation_10k', 'bench_5k'):
    manifest = BASE_JOBS[name]
    predictions = RESULTS / f'{name}.predictions.jsonl'
    run('python', INFER, '--manifest', manifest, '--output-jsonl', predictions,
        '--adapter-dir', ADAPTER, '--audio-root', LOCAL_DATA, '--resume')
    run('python', EVAL, '--predictions-jsonl', predictions, '--output-dir', RESULTS / name)